In [23]:
import matplotlib.style as mplstyle
mplstyle.use('fast')
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import statsmodels.formula.api as smf
import numpy as np
from load_results import load
from scipy.stats import chi2 as chi2_dist
df = load()

In [24]:
A_df = (
    df
    .groupby(["score_name", "base_model", "Experiment", "Rep_A", "Rep_B", "test_type", "k"])["score"]
    .mean()
    .unstack("Experiment")
)

long_df = (
    A_df.reset_index()
     .melt(
         id_vars=["score_name", "base_model", "Rep_A", "Rep_B", "test_type", "k"],
         var_name="dataset",
         value_name="score"
     )
     .rename(columns={"score_name": "metric"})
)
long_df["Rep_pair"] = long_df.apply(
    lambda r: ":".join(sorted([str(r["Rep_A"]), str(r["Rep_B"])])),
    axis=1
)


# long_df = long_df[~((long_df["metric"].isin(["cycle_knn", "cknna", "f1_at_k"])) & (long_df["k"] == 10))] # TODO
# long_df = long_df[~((long_df["metric"].isin(["ndcg", "ndcg_k", "spearman_rho"])) & (long_df["k"].isin([-1,10])))] # TODO
# long_df = long_df[~((long_df["metric"].isin(["rbo"])) & (long_df["k"].isin([10])))] # TODO
# long_df = long_df[~((long_df["metric"].isin(["sign_agreement", "unbiased_cka"])) & (long_df["k"].isin([1,-1,10])))] # TODO
long_df

,metric,base_model,Rep_A,Rep_B,test_type,k,dataset,score,Rep_pair
0,cka,Llama-3.2-1B,$r^{L2}_{\nabla f}$,$r^{id_{X}}_{\nabla f}$,rep,-1,code,1.000000,$r^{L2}_{\nabla f}$:$r^{id_{X}}_{\nabla f}$
1,cka,Llama-3.2-1B,$r^{L2}_{\nabla f}$,$r_{\nabla f}^{aa}$,rep,-1,code,0.866309,$r^{L2}_{\nabla f}$:$r_{\nabla f}^{aa}$
2,cka,Llama-3.2-1B,$r^{L2}_{\nabla f}$,$r_{s_{{test}}}$,rep,-1,code,0.406855,$r^{L2}_{\nabla f}$:$r_{s_{{test}}}$
3,cka,Llama-3.2-1B,$r^{L2}_{\nabla f}$,$r_{{attn}}$,rep,-1,code,0.080853,$r^{L2}_{\nabla f}$:$r_{{attn}}$
4,cka,Llama-3.2-1B,$r^{L2}_{\nabla f}$,$r_{{bm25}}$,rep,-1,code,0.094380,$r^{L2}_{\nabla f}$:$r_{{bm25}}$
...,...,...,...,...,...,...,...,...,...
39895,unbiased_cka,Qwen3-8B-Base,nemot,$r_{{repsim}}$,rep,-1,squad_v2,0.343548,$r_{{repsim}}$:nemot
39896,unbiased_cka,Qwen3-8B-Base,qwen,$r_{{attn}}$,rep,-1,squad_v2,0.024927,$r_{{attn}}$:qwen
39897,unbiased_cka,Qwen3-8B-Base,qwen,$r_{{mp}}$,rep,-1,squad_v2,0.376798,$r_{{mp}}$:qwen
39898,unbiased_cka,Qwen3-8B-Base,qwen,$r_{{repsim}}$,rep,-1,squad_v2,0.109855,$r_{{repsim}}$:qwen


In [25]:
import itertools
import pandas as pd

from schedule_representations import experiments, scripts_gpu, scripts_no_gpu

from schedule_similarity import SCRIPTS, KS, NORMALIZE


def script_label(path):
    return path.split("/")[1].replace("run_", "").replace(".py", "")


def expected_counts():
    """
    Build expected number of rows per group.
    """
    script_pairs = list(itertools.combinations(SCRIPTS, 2))
    num_pairs = len(script_pairs)
    num_configs = len(NORMALIZE) * len(KS)

    expected_per_exp = num_pairs * num_configs

    return expected_per_exp

In [26]:
def test_results(results_df: pd.DataFrame):

    grouped = results_df.groupby(
        ["base_model", "dataset", "k", "test_type"]
    ).size().reset_index(name="count")

    script_pairs = len(list(itertools.combinations(SCRIPTS, 2)))

    for exp in experiments:
        for k in KS:
            for normalize in NORMALIZE:

                expected = script_pairs

                test_type = "normalized" if normalize else "unnormalized"

                row = grouped[
                    (grouped["base_model"] == exp["model"]) &
                    (grouped["dataset"] == exp["dataset"]) &
                    (grouped["k"] == k) &
                    (grouped["test_type"] == test_type)
                ]

                if row.empty:
                    print(f"Missing group: {exp['model']} {k} {test_type}")

                elif int(row["count"].iloc[0]) != expected:
                    print('int(row["count"].iloc[0]) != expected')
test_results(long_df)

Missing group: ./models/Qwen3-1.7B-Base_ftrace_date-lm_conversational_lr0.0001_seed42 -1 normalized
Missing group: ./models/Qwen3-1.7B-Base_ftrace_date-lm_conversational_lr0.0001_seed42 -1 unnormalized
Missing group: ./models/Qwen3-1.7B-Base_tulu-3-sft-personas-math_lr0.0001_seed42 -1 normalized
Missing group: ./models/Qwen3-1.7B-Base_tulu-3-sft-personas-math_lr0.0001_seed42 -1 unnormalized
Missing group: ./models/Qwen3-1.7B-Base_tulu-3-sft-personas-code_lr0.0001_seed42 -1 normalized
Missing group: ./models/Qwen3-1.7B-Base_tulu-3-sft-personas-code_lr0.0001_seed42 -1 unnormalized
Missing group: ./models/Qwen3-1.7B-Base_squad_v2_conversational_lr0.0001_seed42 -1 normalized
Missing group: ./models/Qwen3-1.7B-Base_squad_v2_conversational_lr0.0001_seed42 -1 unnormalized
Missing group: ./models/Qwen3-1.7B-Base_tulu-3-sft-olmo-2-mixture-0225_lr0.0001_seed42 -1 normalized
Missing group: ./models/Qwen3-1.7B-Base_tulu-3-sft-olmo-2-mixture-0225_lr0.0001_seed42 -1 unnormalized
Missing group: ./mod

In [27]:
long_df

,metric,base_model,Rep_A,Rep_B,test_type,k,dataset,score,Rep_pair
0,cka,Llama-3.2-1B,$r^{L2}_{\nabla f}$,$r^{id_{X}}_{\nabla f}$,rep,-1,code,1.000000,$r^{L2}_{\nabla f}$:$r^{id_{X}}_{\nabla f}$
1,cka,Llama-3.2-1B,$r^{L2}_{\nabla f}$,$r_{\nabla f}^{aa}$,rep,-1,code,0.866309,$r^{L2}_{\nabla f}$:$r_{\nabla f}^{aa}$
2,cka,Llama-3.2-1B,$r^{L2}_{\nabla f}$,$r_{s_{{test}}}$,rep,-1,code,0.406855,$r^{L2}_{\nabla f}$:$r_{s_{{test}}}$
3,cka,Llama-3.2-1B,$r^{L2}_{\nabla f}$,$r_{{attn}}$,rep,-1,code,0.080853,$r^{L2}_{\nabla f}$:$r_{{attn}}$
4,cka,Llama-3.2-1B,$r^{L2}_{\nabla f}$,$r_{{bm25}}$,rep,-1,code,0.094380,$r^{L2}_{\nabla f}$:$r_{{bm25}}$
...,...,...,...,...,...,...,...,...,...
39895,unbiased_cka,Qwen3-8B-Base,nemot,$r_{{repsim}}$,rep,-1,squad_v2,0.343548,$r_{{repsim}}$:nemot
39896,unbiased_cka,Qwen3-8B-Base,qwen,$r_{{attn}}$,rep,-1,squad_v2,0.024927,$r_{{attn}}$:qwen
39897,unbiased_cka,Qwen3-8B-Base,qwen,$r_{{mp}}$,rep,-1,squad_v2,0.376798,$r_{{mp}}$:qwen
39898,unbiased_cka,Qwen3-8B-Base,qwen,$r_{{repsim}}$,rep,-1,squad_v2,0.109855,$r_{{repsim}}$:qwen


In [28]:
import itertools

required_pairs = set(itertools.product(long_df["base_model"].unique(),
                                       long_df["dataset"].unique()))

def is_full_cross(sub):
    present = set(zip(sub["base_model"], sub["dataset"]))
    return required_pairs.issubset(present)

long_df = (
    long_df
    .groupby(["metric", "k", "test_type"], group_keys=False)
    .filter(is_full_cross)
)

$y_{i,m,d}=\beta_0+\beta^{(m)}_{m}+\beta^{(d)}_{d}+\beta^{(m \times d)}_{m,d}+u_i+\epsilon_{i,m,d}$

In [29]:
def fit_model(definition, long_df, score_name, k=-1):
    df = long_df[(long_df["metric"] == score_name) & (long_df["k"] == k)].copy()
    cols = ["base_model", "dataset", "score", "Rep_pair"]
    if df[cols].isnull().values.any():
        print(f"missing entries for {score_name} at k={k}")
        # return None
    if (df[cols].nunique() == 1).any():
        print(f"constant entries for {score_name} at k={k}")
        return None
    data = df[cols].dropna().reset_index(drop=True) # TODO remove dropna on full results
    # display(data)
    return smf.mixedlm(
                definition,
                data=data,
                groups=data["Rep_pair"]
            ).fit()

configs = long_df.groupby(["metric", "k", "test_type"]).groups.keys()
models = {
    (score_name, k, test_type) : fit_model("score ~ C(base_model) * C(dataset)", long_df, score_name, k) for score_name, k, test_type in configs
} 

models_no_interaction = {
    (score_name, k, test_type) : fit_model("score ~ C(base_model) + C(dataset)", long_df, score_name, k) for score_name, k, test_type in configs
} 

models_no_base_model = {
    (score_name, k, test_type) : fit_model("score ~ C(dataset)", long_df, score_name, k) for score_name, k, test_type in configs
} 

models_no_dataset = {
    (score_name, k, test_type) : fit_model("score ~ C(base_model)", long_df, score_name, k) for score_name, k, test_type in configs
} 


missing entries for cka at k=-1
missing entries for ndcg at k=-1
missing entries for rbo at k=-1


/usr/local/lib/python3.10/dist-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


missing entries for sign_agreement at k=-1
missing entries for spearman_rho at k=-1
missing entries for unbiased_cka at k=-1
missing entries for cka at k=-1
missing entries for ndcg at k=-1
missing entries for rbo at k=-1


/usr/local/lib/python3.10/dist-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


missing entries for sign_agreement at k=-1
missing entries for spearman_rho at k=-1
missing entries for unbiased_cka at k=-1
missing entries for cka at k=-1
missing entries for ndcg at k=-1
missing entries for rbo at k=-1


/usr/local/lib/python3.10/dist-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


missing entries for sign_agreement at k=-1
missing entries for spearman_rho at k=-1
missing entries for unbiased_cka at k=-1
missing entries for cka at k=-1
missing entries for ndcg at k=-1
missing entries for rbo at k=-1


/usr/local/lib/python3.10/dist-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


missing entries for sign_agreement at k=-1
missing entries for spearman_rho at k=-1
missing entries for unbiased_cka at k=-1


$H_0: \beta^{(m \times d)}_{m,d} = 0 \quad \forall m,d,$



$H_0^{(M)}: \beta^{(m)}_{m} = 0 \quad \forall m$

$H_0^{(D)}: \beta^{(d)}_{d} = 0 \quad \forall d$

In [30]:


def lr_test_df(full_model, reduced_model, term_name):
    lr_stat = 2 * (full_model.llf - reduced_model.llf)
    df_diff = full_model.df_modelwc - reduced_model.df_modelwc
    p_value = chi2_dist.sf(lr_stat, df_diff)

    return pd.DataFrame({
        term_name: [term_name],
        "chi2": [lr_stat],
        "P>chi2": [p_value],
        "df constraint": [df_diff]
    }).set_index(term_name)
def compute_r2_mixedlm(model):
    # fixed effects design matrix
    fe = model.model.exog
    fe_params = model.fe_params.values

    # fixed-effect linear predictor η
    eta = fe @ fe_params
    sigma_f2 = np.var(eta, ddof=1)

    # random intercept variance (Rep_pair)
    sigma_pair2 = model.cov_re.values.item()

    # residual variance
    sigma_e2 = model.scale

    denom = sigma_f2 + sigma_pair2 + sigma_e2

    R2_m = sigma_f2 / denom
    R2_c = (sigma_f2 + sigma_pair2) / denom

    return sigma_f2, sigma_pair2, sigma_e2, R2_m, R2_c


result_dfs_lrt = []
result_dfs_r2 = []
for (score_name, k, test_type), model in models.items():
    if model is None:
        continue
    interaction_test = pd.concat([
        lr_test_df(
            model,
            models_no_base_model[(score_name, k, test_type)],
            "C(base_model)"
        ),
        lr_test_df(
            model,
            models_no_dataset[(score_name, k, test_type)],
            "C(dataset)"
        ),
        lr_test_df(
            model,
            models_no_interaction[(score_name, k, test_type)],
            "C(base_model):C(dataset)"
        )
        ])
    # display(interaction_test)
    df_model_dataset = interaction_test.stack().rename_axis(["term", "stat"]).to_frame().T.reset_index(drop=True)
    df_model_dataset["metric"] = score_name
    df_model_dataset["test_type"] = test_type
    df_model_dataset["k"] = k
    df_model_dataset = df_model_dataset.set_index(["metric", "k"])
    result_dfs_lrt.append(df_model_dataset)
    
    r2_dfs = []
    for setting, model_ in [
        ("full", model),
        ("no_interaction", models_no_interaction[(score_name, k,test_type)]),
        ("no_base_model", models_no_base_model[(score_name, k,test_type)]),
        ("no_dataset", models_no_dataset[(score_name, k,test_type)])
        ]:
        sigma_f2, sigma_pair2, sigma_e2, R2_m, R2_c = compute_r2_mixedlm(model_)
        df_r2 = pd.DataFrame({

            "metric": [score_name],
            "test_type": [test_type],
            "k": [k],
            # "sigma_f2": [sigma_f2],
            # "sigma_pair2": [sigma_pair2],
            # "sigma_e2": [sigma_e2],
            "R2_m": [R2_m],
            "R2_c": [R2_c]
        })
        df_r2.columns = pd.MultiIndex.from_product(
            [[setting], df_r2.columns]
        )

        r2_dfs.append(df_r2)
    result_dfs_r2.append(pd.concat(r2_dfs, axis=1))
        
    

    
 
df_lrt = pd.concat(result_dfs_lrt)
df_r2_model_dataset = pd.concat(result_dfs_r2)




In [31]:
df_display = df_lrt.reset_index().set_index(["test_type", "metric", "k"]).sort_values(["test_type", "metric", "k"])
df_display = df_display.loc[:, df_display.columns.get_level_values("stat") != "df constraint"]

chi2_cols = df_display.columns.get_level_values("stat") == "chi2"


df_display.style.background_gradient(
    cmap="viridis",
    subset=df_display.columns[chi2_cols],
    axis=1
)

In [32]:
df = df_r2_model_dataset.copy()

df = df.set_index(("full", "metric"))

delta_r2 = pd.DataFrame(index=df.index)

delta_r2["interaction_model_dataset_R2_m"] = (
    df[("full", "R2_m")] - df[("no_interaction", "R2_m")]
)
delta_r2["interaction_model_dataset_R2_c"] = (
    df[("full", "R2_c")] - df[("no_interaction", "R2_c")]
)

delta_r2["base_model_R2_m"] = (
    df[("full", "R2_m")] - df[("no_base_model", "R2_m")]
)
delta_r2["base_model_R2_c"] = (
    df[("full", "R2_c")] - df[("no_base_model", "R2_c")]
)

delta_r2["dataset_R2_m"] = (
    df[("full", "R2_m")] - df[("no_dataset", "R2_m")]
)
delta_r2["dataset_R2_c"] = (
    df[("full", "R2_c")] - df[("no_dataset", "R2_c")]
)



r2_m_cols = [col for col in delta_r2.columns if "R2_m" in col]


delta_r2.style.background_gradient(
    cmap="viridis",
    subset=r2_m_cols,
    axis=1
)

,interaction_model_dataset_R2_m,interaction_model_dataset_R2_c,base_model_R2_m,base_model_R2_c,dataset_R2_m,dataset_R2_c
"('full', 'metric')",,,,,,
cka,0.003062,0.002533,0.013132,0.012849,0.021220,0.021042
ndcg,0.001847,0.001728,0.002628,0.002446,0.021017,0.021406
rbo,0.000223,-0.000430,0.000497,-0.000276,0.001900,0.001179
sign_agreement,0.000001,-0.000076,0.000001,-0.000095,0.000063,-0.000025
spearman_rho,0.000208,-0.000540,0.000667,-0.000192,0.001350,0.000536
unbiased_cka,0.002867,0.001301,0.006730,0.005226,0.019649,0.017861


In [33]:
r2_m_cols

['interaction_model_dataset_R2_m', 'base_model_R2_m', 'dataset_R2_m']

### 